# 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Install Libraries

In [ ]:
!pip install -q emoji ftfy nltk transformers torch

# 3. Import Libraries


In [ ]:
import pandas as pd
import re
import emoji
from ftfy import fix_text
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from transformers import pipeline

# 4. Download NLTK Resources

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# 5. Load Dataset

In [ ]:
!find /content/drive/MyDrive -name "raw_data.csv"

/content/drive/MyDrive/Y3S2/HPDP Project/HPDP_PROJECT2/HPDP_PROJECT2_DATA/raw_data.csv
/content/drive/MyDrive/Y3S2/HPDP Project/HPDP_PROJECT/HPDP_PROJECT_DATA/raw_data.csv
/content/drive/MyDrive/raw_data.csv


In [ ]:
DRIVE_FOLDER = '/content/drive/MyDrive/Y3S2/HPDP Project/HPDP_PROJECT2/HPDP_PROJECT2_DATA/'
RAW_PATH = DRIVE_FOLDER + 'raw_data.csv'

df = pd.read_csv(RAW_PATH)

print(df.shape)
df.head()

(3271, 2)


,place_url,review_text
0,https://maps.app.goo.gl/sws8cpK7E1f77P1y6,Batu Caves is an unforgettable blend of cultur...
1,https://maps.app.goo.gl/sws8cpK7E1f77P1y6,"Visited this place in Malaysia, truly it is on..."
2,https://maps.app.goo.gl/sws8cpK7E1f77P1y6,"A must visit if you're in KL, the walk isn't t..."
3,https://maps.app.goo.gl/sws8cpK7E1f77P1y6,"Such an impressive sight on arrival, the large..."
4,https://maps.app.goo.gl/sws8cpK7E1f77P1y6,Batu Caves was first discovered in a way in th...


# 6. Initialize NLP Tools

In [ ]:
stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

# Hugging Face Sentiment Model
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 7. Text Cleaning Function

In [ ]:
def clean_review(text):

    if pd.isna(text):
        return ""

    # Convert to string
    text = str(text)

    # Fix broken encoding
    text = fix_text(text)

    # Convert to lowercase
    text = text.lower()

    # Remove emojis
    text = emoji.replace_emoji(text, replace='')

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove special characters and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Join back into sentence
    cleaned_text = " ".join(tokens)

    return cleaned_text


# 8. Apply Cleaning

In [ ]:
df["clean_review"] = df["review_text"].apply(clean_review)

# 9. Remove Empty Reviews

In [ ]:
df = df[
    df["clean_review"].notna() &
    (df["clean_review"].str.len() > 5)
]

print("After Cleaning:", df.shape)

After Cleaning: (3242, 3)


# 10. Sentiment Labeling using Hugging Face

In [ ]:
def predict_sentiment(text):

    try:
        result = sentiment_pipeline(text[:512])[0]

        label = result['label'].lower()
        score = result['score']

        # Convert labels to numeric values
        if label == "positive":
            sentiment_label = 1

        elif label == "neutral":
            sentiment_label = 0

        else:
            sentiment_label = -1

        return sentiment_label, score

    except:
        return 0, 0.0

# Apply prediction
sentiment_results = df["clean_review"].apply(predict_sentiment)

# Create columns
df["label"] = sentiment_results.apply(lambda x: x[0])
df["confidence"] = sentiment_results.apply(lambda x: x[1])

# 11. Keep Required Columns

In [ ]:
final_df = df[[
    # "place_url",
    "clean_review",
    "label"
    # "confidence"
]]


# 7. Save Clean Dataset to Google Drive

In [ ]:
CLEAN_PATH = DRIVE_FOLDER + 'clean_data.csv'

final_df.to_csv(CLEAN_PATH, index=False)

print(f"Saved to: {CLEAN_PATH}")

final_df.head()

Saved to: /content/drive/MyDrive/Y3S2/HPDP Project/HPDP_PROJECT2/HPDP_PROJECT2_DATA/clean_data.csv


,clean_review,label
0,batu cave unforgettable blend culture spiritua...,1
1,visited place malaysia truly one iconic breath...,1
2,must visit kl walk difficult facility support ...,1
3,impressive sight arrival large statue brightly...,1
4,batu cave first discovered way really much ope...,1
